# Daily box office data

I used this notebook during my thesis to collect daily box office data from The Numbers for the movies in my sample.

The site may have changed since I collected the data, so I am keeping this notebook mainly to show the original collection approach. I have not tested it against the current website.

The input file is a private movie lookup containing the movie title and its The Numbers URL.


In [ ]:
import pandas as pd
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup
import nest_asyncio

nest_asyncio.apply()

CSV_PATH = "../data/private/movie_lookup_with_the_numbers.csv"
OUTPUT_CSV = "../data/private/daily_box_office.csv"

TITLE_COL = "title"
URL_COL = "the_numbers_url"

movies_df = pd.read_csv(CSV_PATH)

print("Loaded", len(movies_df), "movies")
movies_df.head()


## Scrape one movie


In [ ]:
async def scrape_movie(page, movie_name, movie_url, retries=3):
    clean_url = movie_url.split("#")[0]

    for attempt in range(1, retries + 1):
        print(f"Attempt {attempt}: {movie_name}")

        try:
            await page.goto(clean_url, timeout=60000, wait_until="networkidle")
            await page.wait_for_timeout(1500)

            html = await page.content()
            soup = BeautifulSoup(html, "html.parser")

            heading = soup.find(
                "h2",
                string=lambda text: text and "Daily Box Office Performance" in text
            )

            if heading is None:
                raise Exception("Daily box office heading not found")

            # The table I needed was the first table after this heading
            table = heading.find_next("table")

            if table is None:
                raise Exception("Daily box office table not found")

            rows = table.find_all("tr")
            headers = [
                cell.get_text(strip=True)
                for cell in rows[0].find_all(["th", "td"])
            ]
            headers.insert(1, "Date URL")

            data = []

            for row in rows[1:]:
                cells = row.find_all("td")

                if not cells:
                    continue

                values = [cell.get_text(strip=True) for cell in cells]

                # I kept the daily page link as well so individual rows could be checked later
                date_link = cells[0].find("a")
                date_url = None

                if date_link and date_link.has_attr("href"):
                    date_url = "https://www.the-numbers.com" + date_link["href"]

                values.insert(1, date_url)
                data.append(values)

            movie_df = pd.DataFrame(data, columns=headers)
            movie_df["Movie"] = movie_name
            movie_df["Source URL"] = clean_url

            return movie_df

        except Exception as error:
            print("Error:", error)
            await page.wait_for_timeout(attempt * 2000)

    print("Failed:", movie_name)
    return None


## Run the scraper for the movie list


In [ ]:
async def run_batch(movies_df):
    results = []
    errors = []

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        for i, row in movies_df.iterrows():
            movie_name = row[TITLE_COL]
            movie_url = row[URL_COL]

            print(f"\n{i + 1}/{len(movies_df)}: {movie_name}")

            movie_df = await scrape_movie(page, movie_name, movie_url)

            if movie_df is None:
                errors.append({
                    "movie": movie_name,
                    "url": movie_url
                })
            else:
                results.append(movie_df)

        await browser.close()

    if results:
        final_df = pd.concat(results, ignore_index=True)
        final_df.to_csv(OUTPUT_CSV, index=False)
        print("\nSaved:", OUTPUT_CSV)
    else:
        final_df = pd.DataFrame()
        print("\nNo successful scrapes")

    return final_df, pd.DataFrame(errors)


In [ ]:
final_df, error_log = await run_batch(movies_df)

final_df.head()
